# Published results

> **Placeholder:** No canonical timings have been published. Smoke output is diagnostic only.

# Full-workflow simulation scaling

This authoritative notebook replaces the former standalone simulation baseline: `n_zones=1` is the canonical baseline. It covers exactly `[1, 10, 50, 100]` zones using the complete translated 23-component, 32-connection workflow per zone.

## Methodology and environment

Select `smoke` for structural checks or `full` for intentional measurement. Full mode performs five repetitions and retains raw times plus median/spread. Every row records independent `model_layout` (`standard` or `batched`), `execution_mode` (`object` or `functional`), and `execution_backend` (`eager` or `cuda_graph`) fields. CUDA Graph is a backend for functional execution, never an execution mode. Component batching time, functional setup, graph capture, first-call time, and steady replay are reported separately. Checkpoint resume uses these three fields with zone count, device, and repetition. Every zone size receives numerical parity and source-to-batch mapping audits plus topology metadata.

In [ ]:
# Colab bootstrap: configure a branch, tag, or immutable commit SHA.
import os
import pathlib
import subprocess
import sys


GIT_REF = "main"
REPOSITORY = "https://github.com/JBjoernskov/Twin4Build.git"
if "google.colab" in sys.modules:
    root = pathlib.Path("/content/Twin4Build")
    if not root.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(root)], check=True)
    subprocess.run(["git", "-C", str(root), "fetch", "--all", "--tags"], check=True)
    subprocess.run(["git", "-C", str(root), "checkout", GIT_REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(root)], check=True)
else:
    root = pathlib.Path.cwd()
    if root.name == "benchmarks":
        root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from benchmarks.common import (
    BenchmarkConfig,
    SIMULATION_MATRIX,
    run_simulation_matrix,
    ZONE_COUNTS,
    environment_metadata,
    seed_everything,
    serialize_results,
)

config = BenchmarkConfig(mode=os.environ.get("T4B_BENCHMARK_MODE", "smoke"))
seed_everything(config.seed)
assert ZONE_COUNTS == [1, 10, 50, 100]
environment_metadata(GIT_REF)

In [ ]:
print("Zone counts:", ZONE_COUNTS)
print("Matrix:", SIMULATION_MATRIX)
rows = run_simulation_matrix(config)
rows

In [ ]:
result_path = serialize_results("simulation_scaling", config, rows, GIT_REF)
print(result_path)

## Interpretation

Compare rows only when zone count, horizon, precision, device, model layout, execution mode, and backend match the intended comparison. Treat component batching time separately. A functional or CUDA Graph speedup is publishable only when parity passed. The fixed seed, complete topology, five-repeat policy, raw rows, median/spread, and explicit CUDA skips are part of the result contract.